# 14｜Valuation P1 單位與會計期間交叉驗證

只對既有 P0 真實盤點做**唯讀交叉對照**；`RUN_FETCH=false`、`publication_allowed=false`，不改 canonical、不訓練模型。


In [ ]:
from pathlib import Path
import hashlib, importlib, json, sys, zipfile
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/給ai專用/janusChatGPT')
assert ROOT.is_dir(), '不可建立假的 janusChatGPT 根目錄'
CODE = ROOT/'CoLab程式'/'valuation_unit_period_audit.py'
EXPECTED_MODULE_SHA256 = '9dca1f74e60189bbf386848fc862b06283141fb69065c7fa2cb9ba05e6af2e5d'
assert CODE.is_file() and hashlib.sha256(CODE.read_bytes()).hexdigest()==EXPECTED_MODULE_SHA256, 'MODULE_PIN_MISMATCH'
sys.path.insert(0,str(CODE.parent));importlib.invalidate_caches()
from valuation_unit_period_audit import run


In [ ]:
DRY_RUN=True
RUN_FETCH=False
assert DRY_RUN and not RUN_FETCH
P0_ZIP = ROOT/'模型研究'/'Valuation_P0_P1_實測資料與程式_20261024.zip'
assert P0_ZIP.is_file(), '找不到已保存的 P0 有界研究包'
INVENTORY_SHA256='f85fd1c16c286628e02d067088b0f9f473932596d83a098561537cc172bc376c'
WORK=Path('/content/valuation_p1_fixed_input')
WORK.mkdir(exist_ok=True)
INV=WORK/'inventory.csv.gz'
with zipfile.ZipFile(P0_ZIP) as z:
    member='P0_P1_實測報告/inventory.csv.gz'
    with z.open(member) as src, INV.open('wb') as dst:
        import shutil;shutil.copyfileobj(src,dst)
assert hashlib.sha256(INV.read_bytes()).hexdigest()==INVENTORY_SHA256, 'INVENTORY_SHA256_MISMATCH'
print('Frozen P0 inventory SHA verified:', INVENTORY_SHA256)


In [ ]:
OUT=ROOT/'模型研究'/'valuation'/'backfill_candidates'/'P1_20261024'/'unit_period'
SUMMARY=run(INV, OUT, expected_inventory_sha256=INVENTORY_SHA256)
assert SUMMARY['publication_allowed'] is False and SUMMARY['source_unit_verified_rows']==0
print(json.dumps(SUMMARY,ensure_ascii=False,indent=2))


In [ ]:
import pandas as pd
from IPython.display import display
display(pd.read_csv(OUT/'unit_period_cohorts.csv'))
display(pd.read_csv(OUT/'unit_period_pairs.csv').query('status != "CONSISTENT_1000X_COHORT_ONLY"').head(30))
print('Reports:',OUT)


**停止與重跑**：中斷後重新掛載 Drive 再執行即可；固定 P0 ZIP、SHA 與同一 run directory，不需要 API key，不做 HTTP，也不會覆寫 canonical。
